# Workshop 2 · Power BI Dataset Performance · Solution

![Power BI report mockup](../../assets/images/powerbi_report_mockup.png)

This is the reference implementation for the Day 2 serving layer: a monthly aggregate, an incremental-refresh view and a KPI table for Power BI.


## Business Scenario

RetailHub needs fast executive BI and safe drill-through. The solution keeps the Day 1 Gold model intact and adds serving objects optimized for Power BI consumption.


## Target Objects

| Object | Grain | Purpose |
|---|---|---|
| `gold.fact_sales_dashboard_monthly` | month x region x category x channel | Import-friendly summary pages |
| `gold.v_fact_sales_incremental` | one row per line | drill-through and incremental refresh |
| `gold.kpi_monthly` | one row per month | KPI cards and refresh sanity checks |


## Setup

Resolve the participant catalog. `00_setup` executes `USE CATALOG`, so SQL cells can use two-part names such as `gold.fact_sales_dashboard`.


In [ ]:
%run ../../setup/00_setup


### Configuration

Confirm the active catalog and schemas before starting the workshop.


In [ ]:
display(spark.createDataFrame([
    ("CATALOG", CATALOG),
    ("BRONZE", BRONZE),
    ("SILVER", SILVER),
    ("GOLD", GOLD),
], ["Variable", "Value"]))


### Runtime Pre-check

Day 2 starts from the Gold model created in Workshop 1. This check fails early if the model is not available.


In [ ]:
required_tables = [
    f"{GOLD}.dim_date",
    f"{GOLD}.dim_customer",
    f"{GOLD}.dim_product",
    f"{GOLD}.fact_sales",
    f"{GOLD}.fact_sales_dashboard",
]
missing = [t for t in required_tables if not spark.catalog.tableExists(t)]
if missing:
    for table in missing:
        print("[MISSING]", table)
    raise Exception("Pre-check failed. Run Workshop 1 solution before starting Day 2.")
print("[OK] Day 1 Gold model is available.")


## 1. Source Baseline


In [ ]:
%sql
SELECT
  COUNT(*) AS rows,
  COUNT(DISTINCT line_id) AS distinct_lines,
  COUNT(DISTINCT order_id) AS distinct_orders,
  MIN(order_date) AS min_order_date,
  MAX(order_date) AS max_order_date
FROM gold.fact_sales_dashboard


## 2. Build `gold.fact_sales_dashboard_monthly`

The aggregate is intentionally smaller than the line-grain table and keeps the business grain required by executive visuals.


In [ ]:
%sql
CREATE OR REPLACE TABLE gold.fact_sales_dashboard_monthly
COMMENT 'Power BI monthly serving table: completed sales at year_month x customer_region x category x channel grain.'
AS
SELECT
  year_month,
  to_date(concat(year_month, '-01')) AS order_month,
  customer_region,
  category,
  channel,
  COUNT(*) AS line_rows,
  COUNT(DISTINCT order_id) AS completed_orders,
  SUM(quantity) AS quantity,
  ROUND(SUM(line_revenue), 2) AS revenue,
  ROUND(SUM(line_margin), 2) AS gross_margin,
  ROUND(100.0 * SUM(line_margin) / NULLIF(SUM(line_revenue), 0), 2) AS margin_rate_pct
FROM gold.fact_sales_dashboard
WHERE is_completed
GROUP BY year_month, customer_region, category, channel


In [ ]:
%sql
SELECT
  COUNT(*) AS aggregate_rows,
  COUNT(DISTINCT concat_ws('|', year_month, customer_region, category, channel)) AS distinct_grain_keys,
  COUNT(*) - COUNT(DISTINCT concat_ws('|', year_month, customer_region, category, channel)) AS duplicate_grain_keys,
  MIN(year_month) AS min_month,
  MAX(year_month) AS max_month
FROM gold.fact_sales_dashboard_monthly


## 3. Reconcile Aggregate to Detail


In [ ]:
%sql
WITH detail AS (
  SELECT
    ROUND(SUM(line_revenue), 2) AS detail_revenue,
    ROUND(SUM(line_margin), 2) AS detail_margin,
    COUNT(DISTINCT order_id) AS detail_completed_orders
  FROM gold.fact_sales_dashboard
  WHERE is_completed
),
aggregate AS (
  SELECT
    ROUND(SUM(revenue), 2) AS aggregate_revenue,
    ROUND(SUM(gross_margin), 2) AS aggregate_margin,
    SUM(completed_orders) AS aggregate_completed_order_bucket_count
  FROM gold.fact_sales_dashboard_monthly
)
SELECT
  detail_revenue,
  aggregate_revenue,
  ROUND(detail_revenue - aggregate_revenue, 2) AS revenue_diff,
  detail_margin,
  aggregate_margin,
  ROUND(detail_margin - aggregate_margin, 2) AS margin_diff,
  detail_completed_orders,
  aggregate_completed_order_bucket_count
FROM detail
CROSS JOIN aggregate


Note: `aggregate_completed_order_bucket_count` can be higher than the global distinct order count because an order can contribute to multiple category/region/channel buckets. Revenue and margin must reconcile exactly; distinct orders are not additive across arbitrary buckets.


## 4. Build `gold.v_fact_sales_incremental`


In [ ]:
%sql
CREATE OR REPLACE VIEW gold.v_fact_sales_incremental
COMMENT 'Power BI incremental-refresh view over line-grain dashboard table with order_datetime timestamp.'
AS
SELECT
  *,
  CAST(order_date AS TIMESTAMP) AS order_datetime
FROM gold.fact_sales_dashboard


In [ ]:
%sql
SELECT
  COUNT(*) AS rows,
  COUNT(DISTINCT line_id) AS distinct_lines,
  MIN(order_datetime) AS min_order_datetime,
  MAX(order_datetime) AS max_order_datetime
FROM gold.v_fact_sales_incremental


## 5. Test Incremental Refresh Window


In [ ]:
%sql
WITH params AS (
  SELECT
    TIMESTAMP '2024-01-01 00:00:00' AS RangeStart,
    TIMESTAMP '2024-02-01 00:00:00' AS RangeEnd
)
SELECT
  p.RangeStart,
  p.RangeEnd,
  COUNT(*) AS rows_in_window,
  MIN(v.order_date) AS min_order_date,
  MAX(v.order_date) AS max_order_date,
  ROUND(SUM(CASE WHEN v.is_completed THEN v.line_revenue ELSE 0 END), 2) AS completed_revenue
FROM gold.v_fact_sales_incremental v
CROSS JOIN params p
WHERE v.order_datetime >= p.RangeStart
  AND v.order_datetime < p.RangeEnd
GROUP BY p.RangeStart, p.RangeEnd


## 6. Build `gold.kpi_monthly`


In [ ]:
%sql
CREATE OR REPLACE TABLE gold.kpi_monthly
COMMENT 'Monthly KPI table for Power BI cards and refresh sanity checks.'
AS
SELECT
  year_month,
  to_date(concat(year_month, '-01')) AS order_month,
  ROUND(SUM(CASE WHEN is_completed THEN line_revenue ELSE 0 END), 2) AS revenue,
  ROUND(SUM(CASE WHEN is_completed THEN line_margin ELSE 0 END), 2) AS gross_margin,
  COUNT(DISTINCT CASE WHEN is_completed THEN order_id END) AS completed_orders,
  COUNT(DISTINCT CASE WHEN is_returned THEN order_id END) AS returned_orders,
  ROUND(
    100.0 * SUM(CASE WHEN is_completed THEN line_margin ELSE 0 END)
    / NULLIF(SUM(CASE WHEN is_completed THEN line_revenue ELSE 0 END), 0),
    2
  ) AS margin_rate_pct,
  ROUND(
    100.0 * COUNT(DISTINCT CASE WHEN is_returned THEN order_id END)
    / NULLIF(
        COUNT(DISTINCT CASE WHEN is_completed THEN order_id END)
        + COUNT(DISTINCT CASE WHEN is_returned THEN order_id END),
        0
      ),
    2
  ) AS return_rate_pct,
  ROUND(
    SUM(CASE WHEN is_completed THEN line_revenue ELSE 0 END)
    / NULLIF(COUNT(DISTINCT CASE WHEN is_completed THEN order_id END), 0),
    2
  ) AS avg_order_value
FROM gold.fact_sales_dashboard
GROUP BY year_month


In [ ]:
%sql
SELECT
  COUNT(*) AS rows,
  COUNT(DISTINCT year_month) AS distinct_months,
  COUNT(*) - COUNT(DISTINCT year_month) AS duplicate_months,
  MIN(year_month) AS min_month,
  MAX(year_month) AS max_month
FROM gold.kpi_monthly


## 7. KPI Smoke Test


In [ ]:
%sql
SELECT *
FROM gold.kpi_monthly
ORDER BY year_month DESC
LIMIT 12


## 8. BI Contract


In [ ]:
%sql
SELECT
  'gold.fact_sales_dashboard_monthly' AS object_name,
  'year_month x customer_region x category x channel' AS grain,
  'Import' AS recommended_mode,
  'refresh after Gold refresh completes' AS refresh_expectation,
  'executive summary pages and trend visuals' AS primary_use
UNION ALL
SELECT
  'gold.v_fact_sales_incremental',
  'line_id',
  'Import with incremental refresh or controlled DirectQuery',
  'date-window refresh using order_datetime',
  'drill-through and detail validation'
UNION ALL
SELECT
  'gold.kpi_monthly',
  'year_month',
  'Import',
  'refresh after monthly aggregate',
  'KPI cards and refresh sanity checks'


## 9. Quality Gate

This assertion cell is intentionally Python because it turns the workshop checks into a hard stop.


In [ ]:
checks = []
checks_spec = {
    "monthly_duplicate_grain_keys": {
        "tolerance": 0,
        "query": """
            SELECT COUNT(*) - COUNT(DISTINCT concat_ws('|', year_month, customer_region, category, channel)) AS n
            FROM gold.fact_sales_dashboard_monthly
        """,
    },
    "kpi_duplicate_months": {
        "tolerance": 0,
        "query": """
            SELECT COUNT(*) - COUNT(DISTINCT year_month) AS n
            FROM gold.kpi_monthly
        """,
    },
    "incremental_duplicate_lines": {
        "tolerance": 0,
        "query": """
            SELECT COUNT(*) - COUNT(DISTINCT line_id) AS n
            FROM gold.v_fact_sales_incremental
        """,
    },
    "monthly_revenue_diff_abs": {
        "tolerance": 1.00,
        "query": """
            WITH d AS (
              SELECT ROUND(SUM(line_revenue), 2) AS revenue
              FROM gold.fact_sales_dashboard
              WHERE is_completed
            ),
            a AS (
              SELECT ROUND(SUM(revenue), 2) AS revenue
              FROM gold.fact_sales_dashboard_monthly
            )
            SELECT ABS(ROUND(d.revenue - a.revenue, 2)) AS n
            FROM d CROSS JOIN a
        """,
    },
}

for name, spec in checks_spec.items():
    observed = float(spark.sql(spec["query"]).first()["n"] or 0)
    tolerance = float(spec["tolerance"])
    status = "OK" if observed <= tolerance else "FAIL"
    checks.append((name, observed, tolerance, status))

failed = [name for name, observed, tolerance, status in checks if status == "FAIL"]
if failed:
    raise Exception(f"Workshop 2 quality gate failed: {failed}")

display(spark.createDataFrame(checks, ["check_name", "observed_value", "tolerance", "status"]))


## Completion

Workshop 2 serving objects are now ready for Day2_02 performance analysis and Day2_03 AI/BI datasets.
